# Community Detection & Network Analysis with GraphFrames

## Objective
Use Louvain method approximation via GraphFrames' labelPropagation algorithm to:
1. Cluster BIXI stations into communities based on ride flows
2. Calculate inflow/outflow metrics per community per hour
3. Identify station network patterns and travel corridors

## Approach
- Build a directed graph: stations as vertices, rides as weighted edges
- Apply labelPropagation for community detection
- Aggregate statistics by community and time window

In [1]:
import os
import sys
from pathlib import Path

# Add src to path for imports
sys.path.insert(0, str(Path.cwd() / "src"))

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, 
    count, 
    sum as spark_sum,
    hour,
    to_date,
    when,
    broadcast,
    row_number
)
from pyspark.sql.window import Window
from graphframes import GraphFrame
import pandas as pd

# Initialize Spark
spark = SparkSession.builder \
    .appName("bixi-graph-analysis") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.10.0") \
    .getOrCreate()

BASE_PATH = "/home/yuchen/project/bixi-analytics/data"

print("Environment initialized successfully")
print(f"Spark version: {spark.version}")
print(f"Base path: {BASE_PATH}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 10:51:23 WARN Utils: Your hostname, YCLNVO-HOME, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/26 10:51:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/yuchen/project/bixi-analytics/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/yuchen/.ivy2.5.2/cache
The jars for the packages stored in: /home/yuchen/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-18488ee6-52d1-4b3a-aeb7-34eb85b3a4a8;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.10.0 in central
:: resolution report :: resolve 121ms :: artifacts dl 4ms
	:

Environment initialized successfully
Spark version: 4.0.1
Base path: /home/yuchen/project/bixi-analytics/data


In [2]:
# 1. Load Rides Data from Silver Layer
rides_df = spark.read.parquet(f"{BASE_PATH}/silver/rides")

print(f"Total rides loaded: {rides_df.count()}")
rides_df.printSchema()

# Display sample of rides data
rides_df.limit(5).show()

Total rides loaded: 27588048
root
 |-- end_station_key: string (nullable = true)
 |-- start_station_key: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_district: string (nullable = true)
 |-- start_station_latitude: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_district: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_longitude: double (nullable = true)
 |-- start_time_ms: timestamp (nullable = true)
 |-- end_time_ms: timestamp (nullable = true)
 |-- start_station_name_norm: string (nullable = true)
 |-- end_station_name_norm: string (nullable = true)
 |-- start_coord_key: string (nullable = true)
 |-- end_coord_key: string (nullable = true)
 |-- start_canonical_station_id: string (nullable = true)
 |-- end_canonical_station_id: string (nullable = true)
 |-- ride_year: integer (nullable = true)

+

In [14]:
# 2. Prepare Graph Data: Create Tuned Edges (weighted + sparsified)

# Tuning knobs to avoid one giant community in label propagation
min_edge_weight = 300      # Keep only station pairs with at least this many rides
top_k_outgoing = 20       # Keep strongest K outgoing links per source station

# Raw directed ride edges
raw_edges_df = rides_df.select(
    col("start_canonical_station_id").alias("src"),
    col("end_canonical_station_id").alias("dst")
).filter(
    col("src").isNotNull() & col("dst").isNotNull() & (col("src") != col("dst"))
)

# Aggregate to weighted edges
edges_weighted = raw_edges_df.groupBy("src", "dst").agg(
    count("*").alias("weight")
)

# Remove weak links
edges_filtered = edges_weighted.filter(col("weight") >= min_edge_weight)

# Keep only top-K strongest outgoing links per source station
edge_rank_w = Window.partitionBy("src").orderBy(col("weight").desc(), col("dst"))
edges_agg = (
    edges_filtered
    .withColumn("edge_rank", row_number().over(edge_rank_w))
    .filter(col("edge_rank") <= top_k_outgoing)
    .drop("edge_rank")
    .cache()
)

print(f"Raw weighted edges: {edges_weighted.count()}")
print(f"After min_edge_weight >= {min_edge_weight}: {edges_filtered.count()}")
print(f"After top_k_outgoing <= {top_k_outgoing}: {edges_agg.count()}")
edges_agg.orderBy(col("weight").desc()).show(10, truncate=False)

Raw weighted edges: 737703


After min_edge_weight >= 300: 13679


After top_k_outgoing <= 20: 8868
+--------+--------+------+
|src     |dst     |weight|
+--------+--------+------+
|STN_0070|STN_0278|9483  |
|STN_0083|STN_0024|8494  |
|STN_0025|STN_0013|8351  |
|STN_0070|STN_0162|8148  |
|STN_0002|STN_0001|8121  |
|STN_0001|STN_0003|7993  |
|STN_0003|STN_0001|7492  |
|STN_0030|STN_0010|7038  |
|STN_0278|STN_0070|6934  |
|STN_0185|STN_0424|6790  |
+--------+--------+------+
only showing top 10 rows


In [15]:
# 3. Create Vertices (Stations)
# Collect all unique station IDs from both source and destination
vertices_df = (
    rides_df.select(col("start_canonical_station_id").alias("id"))
    .union(rides_df.select(col("end_canonical_station_id").alias("id")))
    .filter(col("id").isNotNull())
    .distinct()
    .cache()
)

print(f"Total unique stations: {vertices_df.count()}")
vertices_df.show(10)

Total unique stations: 1421
+--------+
|      id|
+--------+
|STN_0723|
|STN_0450|
|STN_0507|
|STN_0140|
|STN_0774|
|STN_0373|
|STN_0806|
|STN_0451|
|STN_0487|
|STN_0076|
+--------+
only showing top 10 rows


26/03/26 07:03:31 WARN CacheManager: Asked to cache already cached data.


In [16]:
# 4. Build GraphFrame
graph = GraphFrame(vertices_df, edges_agg)

print(f"Graph Statistics:")
print(f"  Vertices: {graph.vertices.count()}")
print(f"  Edges: {graph.edges.count()}")
print(f"  In-degree stats:")
graph.inDegrees.describe().show()
print(f"  Out-degree stats:")
graph.outDegrees.describe().show()

Graph Statistics:
  Vertices: 1421
  Edges: 8868
  In-degree stats:
+-------+--------+------------------+
|summary|      id|          inDegree|
+-------+--------+------------------+
|  count|     850|               850|
|   mean|    NULL|10.432941176470589|
| stddev|    NULL|15.619695430200446|
|    min|STN_0001|                 1|
|    max|STN_1219|               167|
+-------+--------+------------------+

  Out-degree stats:
+-------+--------+-----------------+
|summary|      id|        outDegree|
+-------+--------+-----------------+
|  count|     928|              928|
|   mean|    NULL|9.556034482758621|
| stddev|    NULL|7.442196249282425|
|    min|STN_0001|                1|
|    max|STN_1215|               20|
+-------+--------+-----------------+



In [17]:
# 5. Community Detection using LabelPropagation
# Label propagation as Louvain-style approximation in GraphFrames

lp_max_iter = 15
communities = graph.labelPropagation(maxIter=lp_max_iter)

print("Community Detection Complete")
print(f"labelPropagation maxIter: {lp_max_iter}")
print(f"Number of communities: {communities.select('label').distinct().count()}")

# Show largest communities first
communities.groupBy("label").agg(
    count("*").alias("station_count")
).orderBy(col("station_count").desc()).show(20, truncate=False)

26/03/26 07:03:42 WARN CacheManager: Asked to cache already cached data.
26/03/26 07:03:43 WARN BlockManager: Block rdd_1002_1 already exists on this machine; not re-adding it
26/03/26 07:03:43 WARN BlockManager: Block rdd_1002_3 already exists on this machine; not re-adding it
26/03/26 07:03:44 WARN LabelPropagation: Returned DataFrame is persistent and materialized!


Community Detection Complete
labelPropagation maxIter: 15
Number of communities: 518
+-----------+-------------+
|label      |station_count|
+-----------+-------------+
|0          |282          |
|2          |144          |
|3          |56           |
|15         |51           |
|16         |51           |
|29         |48           |
|63         |39           |
|68         |33           |
|55         |27           |
|95         |26           |
|40         |24           |
|62         |21           |
|61         |15           |
|17179869257|14           |
|26         |12           |
|58         |12           |
|8589934675 |10           |
|60129542222|10           |
|8589934653 |10           |
|87         |10           |
+-----------+-------------+
only showing top 20 rows


In [18]:
# 6. Calculate Inflow/Outflow per Community per Hour

# Join communities back to rides data
rides_with_communities = (
    rides_df
    .join(
        communities.select(
            col("id").alias("start_canonical_station_id"),
            col("label").alias("start_community")
        ),
        on="start_canonical_station_id",
        how="inner"
    )
    .join(
        communities.select(
            col("id").alias("end_canonical_station_id"),
            col("label").alias("end_community")
        ),
        on="end_canonical_station_id",
        how="inner"
    )
)

# start_time_ms is already TIMESTAMP in silver rides; extract hour directly
rides_with_communities = rides_with_communities.withColumn(
    "hour",
    hour(col("start_time_ms"))
)

# Inter-community ride counts per hour
inflow_outflow = rides_with_communities.groupBy(
    "hour", "start_community", "end_community"
).agg(
    count("*").alias("ride_count")
).cache()

print("Inflow/Outflow by Community and Hour (sample):")
inflow_outflow.show(20, truncate=False)

Inflow/Outflow by Community and Hour (sample):


+----+---------------+-------------+----------+
|hour|start_community|end_community|ride_count|
+----+---------------+-------------+----------+
|1   |0              |2            |45787     |
|8   |40             |0            |12603     |
|10  |0              |0            |317091    |
|4   |16             |0            |37840     |
|3   |16             |0            |29709     |
|4   |0              |2            |55079     |
|9   |61             |61           |1447      |
|6   |0              |0            |708111    |
|7   |15             |0            |12849     |
|5   |2              |2            |268719    |
|4   |63             |63           |36546     |
|13  |0              |0            |112344    |
|0   |26             |81           |463       |
|5   |68             |2            |1988      |
|5   |55             |54           |127       |
|3   |0              |2            |50313     |
|20  |3              |29           |8954      |
|7   |2              |29           |1852

In [19]:
# 7. Aggregate Community Inflow/Outflow Statistics per Hour

# Outflow: rides starting from a community
outflow_by_community_hour = (
    rides_with_communities
    .groupBy("hour", "start_community")
    .agg(count("*").alias("outflow_count"))
    .withColumnRenamed("start_community", "community")
)

# Inflow: rides ending in a community
inflow_by_community_hour = (
    rides_with_communities
    .groupBy("hour", "end_community")
    .agg(count("*").alias("inflow_count"))
    .withColumnRenamed("end_community", "community")
)

# Merge inflow and outflow
community_flow_stats = (
    outflow_by_community_hour
    .join(inflow_by_community_hour, on=["hour", "community"], how="outer")
    .fillna(0)
    .orderBy("hour", "community")
)

print("Community Inflow/Outflow Statistics per Hour:")
community_flow_stats.show(30)

Community Inflow/Outflow Statistics per Hour:


+----+---------+-------------+------------+
|hour|community|outflow_count|inflow_count|
+----+---------+-------------+------------+
|   0|        0|       652126|      598455|
|   0|        2|       223925|      265051|
|   0|        3|       104316|      129118|
|   0|       15|        69379|       71633|
|   0|       16|        63937|       60445|
|   0|       26|         4214|        4238|
|   0|       29|        92530|       93722|
|   0|       40|        36897|       33266|
|   0|       54|         3881|        3953|
|   0|       55|        25054|       22128|
|   0|       58|         5668|        5012|
|   0|       61|         3073|        3092|
|   0|       62|         4421|        4458|
|   0|       63|        37867|       40018|
|   0|       68|        34719|       28522|
|   0|       72|         2331|        2100|
|   0|       79|          901|         977|
|   0|       81|         1722|        1521|
|   0|       87|          806|         832|
|   0|       90|         1396|  

In [20]:
# 8. Community Membership Details
community_summary = (
    communities
    .groupBy("label")
    .agg(
        count("*").alias("station_count"),
        spark_sum(when(col("id").isNotNull(), 1).otherwise(0)).alias("total_stations")
    )
    .withColumnRenamed("label", "community_id")
    .orderBy(col("station_count").desc())
)

print("Community Summary:")
community_summary.show()

# Show top 5 largest communities with station samples
print("\nTop 5 Largest Communities with Station Members:")
top_community_ids = [
    r["community_id"]
    for r in community_summary.select("community_id").limit(5).collect()
]

for comm_id in top_community_ids:
    stations = (
        communities
        .filter(col("label") == comm_id)
        .select("id")
        .orderBy("id")
        .limit(10)
        .collect()
    )
    total = communities.filter(col("label") == comm_id).count()
    print(f"\nCommunity {comm_id} ({total} stations):")
    for station in stations:
        print(f"  - {station['id']}")

Community Summary:
+------------+-------------+--------------+
|community_id|station_count|total_stations|
+------------+-------------+--------------+
|           0|          282|           282|
|           2|          144|           144|
|           3|           56|            56|
|          15|           51|            51|
|          16|           51|            51|
|          29|           48|            48|
|          63|           39|            39|
|          68|           33|            33|
|          55|           27|            27|
|          95|           26|            26|
|          40|           24|            24|
|          62|           21|            21|
|          61|           15|            15|
| 17179869257|           14|            14|
|          26|           12|            12|
|          58|           12|            12|
|  8589934675|           10|            10|
| 60129542222|           10|            10|
|  8589934653|           10|            10|
|          87

## Why Top Communities Are Grouped Together (Behavioral Diagnostics)

### Objective
Explain why the top 4 communities are clustered together based on ride behavior, beyond graph tuning parameters.

### Plan
1. Quantify each top community's internal cohesion vs external mixing.
2. Identify strongest internal and cross-community station corridors.
3. Compare hourly demand signatures (morning/evening concentration and net flow balance).

In [21]:
# Task 1: Internal cohesion vs external mixing for top 4 communities
from pyspark.sql.functions import desc, lit, sum as Fsum

TOP_N = 4

top_communities_df = (
    community_summary
    .select("community_id", "station_count")
    .orderBy(col("station_count").desc())
    .limit(TOP_N)
    .cache()
)

top_ids = [r["community_id"] for r in top_communities_df.collect()]
print(f"Top {TOP_N} community IDs: {top_ids}")

# Outbound rides from top communities, split by internal vs cross-community destination
cohesion_df = (
    rides_with_communities
    .filter(col("start_community").isin(top_ids))
    .withColumn(
        "is_internal",
        when(col("start_community") == col("end_community"), lit(1)).otherwise(lit(0))
    )
    .groupBy("start_community")
    .agg(
        count("*").alias("outbound_rides"),
        Fsum("is_internal").alias("internal_outbound_rides")
    )
    .withColumn("cross_outbound_rides", col("outbound_rides") - col("internal_outbound_rides"))
    .withColumn("internal_share", col("internal_outbound_rides") / col("outbound_rides"))
    .withColumnRenamed("start_community", "community_id")
    .join(top_communities_df, on="community_id", how="left")
    .orderBy(desc("station_count"))
)

print("Top community cohesion diagnostics:")
cohesion_df.show(TOP_N, truncate=False)

Top 4 community IDs: [0, 2, 3, 15]
Top community cohesion diagnostics:


+------------+--------------+-----------------------+--------------------+-------------------+-------------+
|community_id|outbound_rides|internal_outbound_rides|cross_outbound_rides|internal_share     |station_count|
+------------+--------------+-----------------------+--------------------+-------------------+-------------+
|0           |12353754      |8277424                |4076330             |0.6700330927748763 |282          |
|2           |4117255       |2587232                |1530023             |0.6283876029053338 |144          |
|3           |2118238       |796640                 |1321598             |0.37608616217818774|56           |
|15          |1268516       |663354                 |605162              |0.5229370382399592 |51           |
+------------+--------------+-----------------------+--------------------+-------------------+-------------+



In [22]:
# Task 2: Strongest internal and cross-community corridors among top communities
from pyspark.sql.window import Window

# Station ID -> most common observed station name (from start/end records)
station_name_freq = (
    rides_df.select(
        col("start_canonical_station_id").alias("station_id"),
        col("start_station_name").alias("station_name")
    )
    .union(
        rides_df.select(
            col("end_canonical_station_id").alias("station_id"),
            col("end_station_name").alias("station_name")
        )
    )
    .filter(col("station_id").isNotNull() & col("station_name").isNotNull())
    .groupBy("station_id", "station_name")
    .agg(count("*").alias("name_freq"))
)

name_rank_w = Window.partitionBy("station_id").orderBy(col("name_freq").desc(), col("station_name"))
station_lookup = (
    station_name_freq
    .withColumn("rn", row_number().over(name_rank_w))
    .filter(col("rn") == 1)
    .select("station_id", "station_name")
)

corridors_top4 = (
    rides_with_communities
    .filter(col("start_community").isin(top_ids) & col("end_community").isin(top_ids))
    .groupBy(
        "start_community", "end_community",
        "start_canonical_station_id", "end_canonical_station_id"
    )
    .agg(count("*").alias("ride_count"))
)

# Top internal corridors per top community
internal_rank_w = Window.partitionBy("start_community").orderBy(col("ride_count").desc())
internal_top = (
    corridors_top4
    .filter(col("start_community") == col("end_community"))
    .withColumn("rk", row_number().over(internal_rank_w))
    .filter(col("rk") <= 5)
    .join(station_lookup.withColumnRenamed("station_id", "start_canonical_station_id")
                      .withColumnRenamed("station_name", "start_station_name_best"),
          on="start_canonical_station_id", how="left")
    .join(station_lookup.withColumnRenamed("station_id", "end_canonical_station_id")
                      .withColumnRenamed("station_name", "end_station_name_best"),
          on="end_canonical_station_id", how="left")
    .select(
        "start_community",
        "start_canonical_station_id", "start_station_name_best",
        "end_canonical_station_id", "end_station_name_best",
        "ride_count", "rk"
    )
    .orderBy("start_community", "rk")
)

# Top cross-community corridors among top communities
cross_top = (
    corridors_top4
    .filter(col("start_community") != col("end_community"))
    .orderBy(col("ride_count").desc())
    .limit(15)
    .join(station_lookup.withColumnRenamed("station_id", "start_canonical_station_id")
                      .withColumnRenamed("station_name", "start_station_name_best"),
          on="start_canonical_station_id", how="left")
    .join(station_lookup.withColumnRenamed("station_id", "end_canonical_station_id")
                      .withColumnRenamed("station_name", "end_station_name_best"),
          on="end_canonical_station_id", how="left")
    .select(
        "start_community", "end_community",
        "start_canonical_station_id", "start_station_name_best",
        "end_canonical_station_id", "end_station_name_best",
        "ride_count"
    )
)

print("Top internal corridors (top 5 per community):")
internal_top.show(50, truncate=False)
print("Top cross-community corridors (top 15 overall among top 4 communities):")
cross_top.show(15, truncate=False)

Top internal corridors (top 5 per community):


+---------------+--------------------------+-----------------------------------------------+------------------------+-----------------------------------------------+----------+---+
|start_community|start_canonical_station_id|start_station_name_best                        |end_canonical_station_id|end_station_name_best                          |ride_count|rk |
+---------------+--------------------------+-----------------------------------------------+------------------------+-----------------------------------------------+----------+---+
|0              |STN_0025                  |Berri / Cherrier                               |STN_0013                |Émile-Duployé / Sherbrooke                     |8351      |1  |
|0              |STN_0002                  |du Mont-Royal / Clark                          |STN_0001                |Métro Mont-Royal (Utilités publiques / Rivard) |8121      |2  |
|0              |STN_0001                  |Métro Mont-Royal (Utilités publiques / Rivard) |STN

+---------------+-------------+--------------------------+----------------------------------------+------------------------+------------------------------------------------------+----------+
|start_community|end_community|start_canonical_station_id|start_station_name_best                 |end_canonical_station_id|end_station_name_best                                 |ride_count|
+---------------+-------------+--------------------------+----------------------------------------+------------------------+------------------------------------------------------+----------+
|0              |3            |STN_0013                  |Émile-Duployé / Sherbrooke              |STN_0010                |Métro Papineau (Dorion / De Maisonneuve)              |3318      |
|15             |3            |STN_0044                  |Métro Frontenac (Ontario / du Havre)    |STN_0216                |Fullum / de Rouen                                     |2930      |
|3              |15           |STN_0216      

In [23]:
# Task 3: Hourly demand signatures for top communities
from pyspark.sql.functions import avg, abs as Fabs

hourly_top = (
    community_flow_stats
    .filter(col("community").isin(top_ids))
    .withColumn("net_flow", col("inflow_count") - col("outflow_count"))
    .cache()
)

signature_df = (
    hourly_top
    .groupBy("community")
    .agg(
        Fsum("outflow_count").alias("total_outflow"),
        Fsum("inflow_count").alias("total_inflow"),
        Fsum(when(col("hour").between(7, 10), col("outflow_count")).otherwise(lit(0))).alias("morning_outflow"),
        Fsum(when(col("hour").between(16, 19), col("outflow_count")).otherwise(lit(0))).alias("evening_outflow"),
        Fsum(when(col("hour").between(7, 10), col("inflow_count")).otherwise(lit(0))).alias("morning_inflow"),
        Fsum(when(col("hour").between(16, 19), col("inflow_count")).otherwise(lit(0))).alias("evening_inflow"),
        avg(Fabs(col("net_flow"))).alias("avg_abs_hourly_imbalance")
    )
    .withColumn("morning_out_share", col("morning_outflow") / col("total_outflow"))
    .withColumn("evening_out_share", col("evening_outflow") / col("total_outflow"))
    .withColumn("net_total", col("total_inflow") - col("total_outflow"))
    .orderBy(col("total_outflow").desc())
)

print("Hourly signature summary for top communities:")
signature_df.select(
    "community", "total_outflow", "total_inflow", "net_total",
    "morning_out_share", "evening_out_share", "avg_abs_hourly_imbalance"
).show(TOP_N, truncate=False)

print("Sample hourly profile (first 40 rows):")
hourly_top.orderBy("community", "hour").show(40, truncate=False)

Hourly signature summary for top communities:


+---------+-------------+------------+---------+-------------------+--------------------+------------------------+
|community|total_outflow|total_inflow|net_total|morning_out_share  |evening_out_share   |avg_abs_hourly_imbalance|
+---------+-------------+------------+---------+-------------------+--------------------+------------------------+
|0        |12353754     |11199032    |-1154722 |0.20454737887770794|0.0488236207390887  |48113.416666666664      |
|2        |4117255      |4765829     |648574   |0.2049078815861539 |0.040951556316040666|27879.666666666668      |
|3        |2118238      |2604396     |486158   |0.2241079614283192 |0.044926018700448205|20965.333333333332      |
|15       |1268516      |1440269     |171753   |0.19668494524310295|0.06692308177429374 |8818.958333333334       |
+---------+-------------+------------+---------+-------------------+--------------------+------------------------+

Sample hourly profile (first 40 rows):
+----+---------+-------------+----------

In [24]:
# Compact summary: top community behavior drivers
summary_rows = (
    signature_df
    .select(
        "community", "total_outflow", "total_inflow", "net_total",
        "morning_out_share", "evening_out_share", "avg_abs_hourly_imbalance"
    )
    .orderBy(col("total_outflow").desc())
    .collect()
)

print("Top community behavior summary:")
for r in summary_rows:
    print(
        f"community={r['community']}, out={r['total_outflow']}, in={r['total_inflow']}, "
        f"net={r['net_total']}, morning_share={float(r['morning_out_share']):.3f}, "
        f"evening_share={float(r['evening_out_share']):.3f}, "
        f"avg_hourly_imbalance={float(r['avg_abs_hourly_imbalance']):.1f}"
    )

Top community behavior summary:
community=0, out=12353754, in=11199032, net=-1154722, morning_share=0.205, evening_share=0.049, avg_hourly_imbalance=48113.4
community=2, out=4117255, in=4765829, net=648574, morning_share=0.205, evening_share=0.041, avg_hourly_imbalance=27879.7
community=3, out=2118238, in=2604396, net=486158, morning_share=0.224, evening_share=0.045, avg_hourly_imbalance=20965.3
community=15, out=1268516, in=1440269, net=171753, morning_share=0.197, evening_share=0.067, avg_hourly_imbalance=8819.0


In [26]:
# Keep Spark session alive for parameter sweeps.
# Run spark.stop() only when all experiments are finished.
print("Spark session kept active for evaluation sweeps.")

ConnectionRefusedError: [Errno 111] Connection refused

## Settings Sweep With Quality Evaluation

### Objective
Reduce oversized communities while preserving meaningful grouping quality.

### What We Evaluate Per Setting
- `largest_share`: fraction of stations in the largest community (smaller is better)
- `internal_share`: fraction of rides staying within the same community (higher is better)
- `n_communities`: total community count (avoid extreme fragmentation)
- `singleton_share`: share of 1-station communities (lower is better)

### Practical Rule
Prefer settings with lower `largest_share` **without** a sharp drop in `internal_share` or a spike in `singleton_share`.

In [2]:
# Task 1: Memory-safe evaluator for community settings
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from graphframes import GraphFrame

# Confirm Spark session is alive
spark.range(1).count()

# Use a partition-pruned subset for tuning in local mode, then rerun best config on full data.
eval_year = 2025
sample_frac = 0.20

rides_eval = (
    spark.read.parquet(f"{BASE_PATH}/silver/rides/ride_year={eval_year}")
    .sample(withReplacement=False, fraction=sample_frac, seed=42)
    .cache()
)
_ = rides_eval.count()

vertices_eval = (
    rides_eval.select(F.col("start_canonical_station_id").alias("id"))
    .union(rides_eval.select(F.col("end_canonical_station_id").alias("id")))
    .filter(F.col("id").isNotNull())
    .distinct()
    .cache()
)
station_count_total = vertices_eval.count()

base_edges = (
    rides_eval.select(
        F.col("start_canonical_station_id").alias("src"),
        F.col("end_canonical_station_id").alias("dst")
    )
    .filter(F.col("src").isNotNull() & F.col("dst").isNotNull() & (F.col("src") != F.col("dst")))
    .cache()
)
_ = base_edges.count()

print(f"Evaluation year: {eval_year}")
print(f"Sample fraction: {sample_frac}")
print(f"Stations in sampled universe: {station_count_total}")


def evaluate_setting(min_edge_weight: int, top_k_outgoing: int, lp_max_iter: int = 12):
    edges_weighted_local = base_edges.groupBy("src", "dst").agg(F.count("*").alias("weight"))
    edges_filtered_local = edges_weighted_local.filter(F.col("weight") >= F.lit(min_edge_weight))

    rank_w = Window.partitionBy("src").orderBy(F.col("weight").desc(), F.col("dst"))
    edges_tuned_local = (
        edges_filtered_local
        .withColumn("rk", F.row_number().over(rank_w))
        .filter(F.col("rk") <= F.lit(top_k_outgoing))
        .drop("rk")
    )

    active_vertices = (
        edges_tuned_local.select(F.col("src").alias("id"))
        .union(edges_tuned_local.select(F.col("dst").alias("id")))
        .distinct()
    )

    active_station_count = active_vertices.count()
    if active_station_count == 0:
        return {
            "min_edge_weight": min_edge_weight,
            "top_k_outgoing": top_k_outgoing,
            "lp_max_iter": lp_max_iter,
            "active_stations": 0,
            "coverage_share": 0.0,
            "n_communities": 0,
            "largest_community": 0,
            "largest_share": 0.0,
            "singleton_communities": 0,
            "singleton_share": 0.0,
            "internal_share": 0.0,
            "score": -1.0,
        }

    graph_local = GraphFrame(active_vertices, edges_tuned_local)
    communities_local = graph_local.labelPropagation(maxIter=lp_max_iter)

    sizes = communities_local.groupBy("label").agg(F.count("*").alias("station_count"))
    n_communities = sizes.count()
    largest = sizes.agg(F.max("station_count").alias("m")).first()["m"]
    singletons = sizes.filter(F.col("station_count") == 1).count()

    rides_with_comm_local = (
        rides_eval
        .join(
            communities_local.select(F.col("id").alias("start_canonical_station_id"), F.col("label").alias("sc")),
            on="start_canonical_station_id", how="inner"
        )
        .join(
            communities_local.select(F.col("id").alias("end_canonical_station_id"), F.col("label").alias("ec")),
            on="end_canonical_station_id", how="inner"
        )
    )

    total_rides_eval = rides_with_comm_local.count()
    internal_rides = rides_with_comm_local.filter(F.col("sc") == F.col("ec")).count()

    internal_share = (internal_rides / total_rides_eval) if total_rides_eval else 0.0
    largest_share = (largest / active_station_count) if active_station_count else 0.0
    singleton_share = (singletons / n_communities) if n_communities else 0.0
    coverage_share = (active_station_count / station_count_total) if station_count_total else 0.0

    score = internal_share + 0.2 * coverage_share - 0.6 * largest_share - 0.15 * singleton_share

    return {
        "min_edge_weight": min_edge_weight,
        "top_k_outgoing": top_k_outgoing,
        "lp_max_iter": lp_max_iter,
        "active_stations": active_station_count,
        "coverage_share": coverage_share,
        "n_communities": n_communities,
        "largest_community": largest,
        "largest_share": largest_share,
        "singleton_communities": singletons,
        "singleton_share": singleton_share,
        "internal_share": internal_share,
        "score": score,
    }

Evaluation year: 2025
Sample fraction: 0.2
Stations in sampled universe: 1238


In [4]:
# Task 2: Targeted sweep for balanced community quality

settings_grid = [
    {"min_edge_weight": 40, "top_k_outgoing": 30, "lp_max_iter": 12},
    {"min_edge_weight": 60, "top_k_outgoing": 25, "lp_max_iter": 12},
    {"min_edge_weight": 80, "top_k_outgoing": 20, "lp_max_iter": 12},
    {"min_edge_weight": 120, "top_k_outgoing": 20, "lp_max_iter": 12},
    {"min_edge_weight": 150, "top_k_outgoing": 15, "lp_max_iter": 12},
    {"min_edge_weight": 300, "top_k_outgoing": 20, "lp_max_iter": 12},
]

results = []
for cfg in settings_grid:
    print(f"Evaluating: {cfg}")
    out = evaluate_setting(**cfg)
    results.append(out)

results_df = spark.createDataFrame(results)

print("\nAll evaluated settings (ranked):")
results_df.orderBy(F.col("score").desc()).show(truncate=False)

# Softer but practical guardrails for local-mode tuning
candidates_df = (
    results_df
    .filter(F.col("largest_share") <= 0.45)
    .filter(F.col("internal_share") >= 0.22)
    .filter(F.col("coverage_share") >= 0.25)
    .orderBy(F.col("score").desc())
)

print("\nCandidate settings:")
candidates_df.show(truncate=False)

best = candidates_df.limit(1).collect()
if best:
    b = best[0]
    print("\nRecommended next setting:")
    print(
        f"min_edge_weight={b['min_edge_weight']}, "
        f"top_k_outgoing={b['top_k_outgoing']}, "
        f"lp_max_iter={b['lp_max_iter']}"
    )
else:
    print("\nNo candidate passed all guardrails. Try lowering min_edge_weight further.")

Evaluating: {'min_edge_weight': 40, 'top_k_outgoing': 30, 'lp_max_iter': 12}


26/03/26 07:15:58 WARN BlockManager: Block rdd_2557_0 already exists on this machine; not re-adding it
26/03/26 07:16:00 WARN LabelPropagation: Returned DataFrame is persistent and materialized!


Evaluating: {'min_edge_weight': 60, 'top_k_outgoing': 25, 'lp_max_iter': 12}


26/03/26 07:16:04 WARN TaskMemoryManager: Failed to allocate a page (2097152 bytes), try again.


[97.956s][warning][gc,alloc] Executor task launch worker for task 2.0 in stage 3845.0 (TID 3813): Retried waiting for GCLocker too often allocating 262146 words


...

Evaluating: {'min_edge_weight': 150, 'top_k_outgoing': 15, 'lp_max_iter': 12}


26/03/26 07:16:40 WARN CacheManager: Asked to cache already cached data.        
26/03/26 07:16:48 WARN LabelPropagation: Returned DataFrame is persistent and materialized!


Evaluating: {'min_edge_weight': 300, 'top_k_outgoing': 20, 'lp_max_iter': 12}


26/03/26 07:16:52 WARN CacheManager: Asked to cache already cached data.        
26/03/26 07:16:59 WARN LabelPropagation: Returned DataFrame is persistent and materialized!



All evaluated settings (ranked):
+---------------+-------------------+-------------------+-----------------+-------------------+-----------+---------------+-------------+-------------------+---------------------+-------------------+--------------+
|active_stations|coverage_share     |internal_share     |largest_community|largest_share      |lp_max_iter|min_edge_weight|n_communities|score              |singleton_communities|singleton_share    |top_k_outgoing|
+---------------+-------------------+-------------------+-----------------+-------------------+-----------+---------------+-------------+-------------------+---------------------+-------------------+--------------+
|872            |0.7043618739903069 |0.607306673098039  |308              |0.3532110091743119 |12         |40             |46           |0.4840785293480349 |16                   |0.34782608695652173|30            |
|763            |0.6163166397415186 |0.497917624882417  |190              |0.2490170380078637 |12         

## Station-Hour Flow First, Then Community Join

### Objective
Compute station-level hourly inflow/outflow directly from raw rides files, then attach community labels and keep only stations in the largest detected community.

### Plan
1. Read raw rides and build `outflow_hourly_sdf` and `inflow_hourly_sdf` by `canonical_station_id` and `ts_hour`.
2. Full-join into station-hour flow table with `inflow`, `outflow`, and `net_flow`.
3. Find the largest community in `communities_rec` and filter to those stations after joining labels.

In [2]:
# Task 1 (corrected): station-hour inflow/outflow from raw rides, then join communities_rec
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from graphframes import GraphFrame

# Read raw rides files from silver; set to None to include all years.
feature_year = 2025
if feature_year is None:
    rides_feature_base = spark.read.parquet(f"{BASE_PATH}/silver/rides")
else:
    rides_feature_base = spark.read.parquet(f"{BASE_PATH}/silver/rides/ride_year={feature_year}")

# Rebuild communities_rec only if it is not available in the current kernel state.
if "communities_rec" not in globals():
    rec_min_edge_weight = 40
    rec_top_k_outgoing = 30
    rec_lp_max_iter = 12

    feature_edges_base = (
        rides_feature_base.select(
            F.col("start_canonical_station_id").alias("src"),
            F.col("end_canonical_station_id").alias("dst")
        )
        .filter(F.col("src").isNotNull() & F.col("dst").isNotNull() & (F.col("src") != F.col("dst")))
    )

    feature_edges_weighted = feature_edges_base.groupBy("src", "dst").agg(F.count("*").alias("weight"))
    feature_edges_filtered = feature_edges_weighted.filter(F.col("weight") >= F.lit(rec_min_edge_weight))

    feature_rank_w = Window.partitionBy("src").orderBy(F.col("weight").desc(), F.col("dst"))
    feature_edges_tuned = (
        feature_edges_filtered
        .withColumn("rk", F.row_number().over(feature_rank_w))
        .filter(F.col("rk") <= F.lit(rec_top_k_outgoing))
        .drop("rk")
    )

    feature_vertices = (
        feature_edges_tuned.select(F.col("src").alias("id"))
        .union(feature_edges_tuned.select(F.col("dst").alias("id")))
        .distinct()
    )

    feature_graph = GraphFrame(feature_vertices, feature_edges_tuned)
    communities_rec = feature_graph.labelPropagation(maxIter=rec_lp_max_iter)

# Same pattern as 05_timeseries_eda Cell 20, generalized to all stations.
outflow_hourly_sdf = (
    rides_feature_base
    .where(F.col("start_canonical_station_id").isNotNull())
    .where(F.col("start_time_ms").isNotNull())
    .groupBy(
        F.col("start_canonical_station_id").alias("canonical_station_id"),
        F.date_trunc("hour", F.col("start_time_ms")).alias("ts_hour")
    )
    .agg(F.count("*").alias("outflow"))
)

inflow_hourly_sdf = (
    rides_feature_base
    .where(F.col("end_canonical_station_id").isNotNull())
    .where(F.col("end_time_ms").isNotNull())
    .groupBy(
        F.col("end_canonical_station_id").alias("canonical_station_id"),
        F.date_trunc("hour", F.col("end_time_ms")).alias("ts_hour")
    )
    .agg(F.count("*").alias("inflow"))
)

station_hourly_flow = (
    outflow_hourly_sdf.join(
        inflow_hourly_sdf,
        on=["canonical_station_id", "ts_hour"],
        how="full"
    )
    .fillna(0, subset=["outflow", "inflow"])
    .withColumn("net_flow", F.col("inflow") - F.col("outflow"))
)

# Keep only stations in largest community after joining labels.
top_community_row = (
    communities_rec.groupBy("label")
    .agg(F.count("*").alias("station_count"))
    .orderBy(F.col("station_count").desc(), F.col("label"))
    .limit(1)
    .collect()[0]
)
top_community_label = top_community_row["label"]
top_community_station_count = top_community_row["station_count"]

top_community_stations = (
    communities_rec
    .filter(F.col("label") == F.lit(top_community_label))
    .select(F.col("id").alias("canonical_station_id"), F.col("label").alias("community"))
)

station_hourly_top_community = (
    station_hourly_flow
    .join(top_community_stations, on="canonical_station_id", how="inner")
    .select("canonical_station_id", "community", "ts_hour", "inflow", "outflow", "net_flow")
)

print("Corrected flow complete: station-hour first, then community join")
print(f"feature_year={feature_year}")
print(f"top_community_label={top_community_label}, station_count={top_community_station_count}")
print(f"station_hourly_flow rows={station_hourly_flow.count()}")
print(f"station_hourly_top_community rows={station_hourly_top_community.count()}")
station_hourly_top_community.limit(12).show(truncate=False)

26/03/26 10:51:43 WARN LabelPropagation: Returned DataFrame is persistent and materialized!


Corrected flow complete: station-hour first, then community join
feature_year=2025
top_community_label=0, station_count=250


station_hourly_flow rows=3860155


station_hourly_top_community rows=1112347


+--------------------+---------+-------------------+------+-------+--------+
|canonical_station_id|community|ts_hour            |inflow|outflow|net_flow|
+--------------------+---------+-------------------+------+-------+--------+
|STN_0001            |0        |2025-01-02 05:00:00|4     |2      |2       |
|STN_0001            |0        |2025-01-02 09:00:00|1     |2      |-1      |
|STN_0001            |0        |2025-01-02 19:00:00|2     |0      |2       |
|STN_0001            |0        |2025-01-04 06:00:00|2     |7      |-5      |
|STN_0001            |0        |2025-01-04 23:00:00|2     |8      |-6      |
|STN_0001            |0        |2025-01-05 00:00:00|1     |1      |0       |
|STN_0001            |0        |2025-01-05 13:00:00|1     |1      |0       |
|STN_0001            |0        |2025-01-05 16:00:00|2     |4      |-2      |
|STN_0001            |0        |2025-01-06 00:00:00|3     |1      |2       |
|STN_0001            |0        |2025-01-06 12:00:00|1     |1      |0       |

In [3]:
# Task 2 (revised): station-hour rows enriched with community-hour features

# Station-level naming convention.
station_hourly_features = (
    station_hourly_top_community
    .select(
        "canonical_station_id",
        "community",
        "ts_hour",
        F.col("inflow").alias("station_inflow"),
        F.col("outflow").alias("station_outflow"),
        F.col("net_flow").alias("station_net_flow")
    )
)

# Community-hour features (from station rows) with explicit community_* names.
community_hourly_features = (
    station_hourly_features
    .groupBy("community", "ts_hour")
    .agg(
        F.countDistinct("canonical_station_id").alias("community_active_stations"),
        F.sum("station_inflow").alias("community_inflow"),
        F.sum("station_outflow").alias("community_outflow"),
        F.sum("station_net_flow").alias("community_net_flow")
    )
)

# Join on (community, ts_hour) to attach community context to every station-hour row.
station_community_hourly_features = (
    station_hourly_features
    .join(community_hourly_features, on=["community", "ts_hour"], how="left")
)

print("Created station+community hourly features with explicit naming")
print("station_community_hourly_features sample:")
station_community_hourly_features.limit(12).show(truncate=False)

print("community_hourly_features sample:")
community_hourly_features.limit(12).show(truncate=False)

Created station+community hourly features with explicit naming
station_community_hourly_features sample:


[44.116s][warning][gc,alloc] Executor task launch worker for task 1.0 in stage 566.0 (TID 491): Retried waiting for GCLocker too often allocating 1048578 words


26/03/26 10:52:08 WARN TaskMemoryManager: Failed to allocate a page (8388608 bytes), try again.


+---------+-------------------+--------------------+--------------+---------------+----------------+-------------------------+----------------+-----------------+------------------+
|community|ts_hour            |canonical_station_id|station_inflow|station_outflow|station_net_flow|community_active_stations|community_inflow|community_outflow|community_net_flow|
+---------+-------------------+--------------------+--------------+---------------+----------------+-------------------------+----------------+-----------------+------------------+
|0        |2025-01-02 09:00:00|STN_0001            |1             |2              |-1              |28                       |26              |29               |-3                |
|0        |2025-01-04 23:00:00|STN_0001            |2             |8              |-6              |35                       |48              |59               |-11               |
|0        |2025-01-06 16:00:00|STN_0001            |2             |2              |0           

+---------+-------------------+-------------------------+----------------+-----------------+------------------+
|community|ts_hour            |community_active_stations|community_inflow|community_outflow|community_net_flow|
+---------+-------------------+-------------------------+----------------+-----------------+------------------+
|0        |2025-01-02 09:00:00|28                       |26              |29               |-3                |
|0        |2025-03-19 21:00:00|64                       |156             |181              |-25               |
|0        |2025-04-15 03:00:00|204                      |619             |697              |-78               |
|0        |2025-04-30 13:00:00|106                      |124             |138              |-14               |
|0        |2025-05-29 00:00:00|207                      |1289            |1399             |-110              |
|0        |2025-05-31 06:00:00|208                      |2462            |2550             |-88         